# IAI600 — Laboratory I: Solution
---

## Part I — Setting up the Environment

Import the required numerical, data manipulation, plotting, and Scikit-Learn tools.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

plt.rcParams["figure.figsize"] = (10, 6)
print("Environment configured successfully.")

from pathlib import Path

## Part II — Obtain and Load the Dataset

In [ ]:
# Load and recreate train/test split from Lab 1
candidates = [
    Path("datasets/housing/housing.csv"),
    Path("datasets/housing.csv"),
    Path("housing.csv"),
]
housing_file = next((path for path in candidates if path.is_file()), None)

if housing_file is None:
    raise FileNotFoundError("Could not find housing.csv. Check the dataset location.")

housing = pd.read_csv(housing_file)


display(housing.head())
housing.info()
display(housing.describe())

### Task 2.1 — Understand the Dataset

1. **Observations:** 20,640 district rows.
2. **Attributes:** 10 columns.
3. **Numerical attributes:** `longitude`, `latitude`, `housing_median_age`, `total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`, and `median_house_value`.
4. **Categorical attribute:** `ocean_proximity` (text).
5. **Target variable:** `median_house_value`.
6. **Missing values:** Yes; `total_bedrooms` has 20,433 non-null entries (207 missing values).

## Part III — Exploring the Data

In [ ]:
housing.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

### Task 3.1 — Attribute Distributions

- **Approximately Symmetric:** `housing_median_age` is the closest to symmetric (though capped at the end).
- **Strongly Skewed:** `total_rooms`, `total_bedrooms`, `population`, `households`, and `median_income` are heavily right-skewed with long right tails.
- **Different Scales:** Attributes vary greatly—`median_income` ranges from ~0.5 to 15, while `population` goes up to ~35,000.
- **Unusual Values:** `housing_median_age` (capped at 52) and `median_house_value` (capped at 500,001) display tall boundary spikes due to upper limit truncation.

## Part IV — Creating a Test Set

In [ ]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)
print(f"Training set: {len(train_set)} rows | Test set: {len(test_set)} rows")

### Task 4.1 — Why Separate the Data?

1. **Why a test set?** It evaluates model generalization on unseen data, preventing fake optimism caused by memorizing training instances.
2. **Repeated use risk:** Repeatedly testing during model design leads to *data snooping bias*, where hyperparameter choices implicitly overfit to the test set.
3. **Learning preprocessing from train data:** Calculating parameters (like median or mean) on the entire dataset leaks test set distribution info into the model (*data leakage*).

## Part V — Stratified Sampling

In [ ]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# Compare sampling distributions
overall = housing["income_cat"].value_counts(normalize=True).sort_index()
strat = strat_test_set["income_cat"].value_counts(normalize=True).sort_index()
rand = pd.cut(
    test_set["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
).value_counts(normalize=True).sort_index()

comp_df = pd.DataFrame({"Overall": overall, "Stratified": strat, "Random": rand})
comp_df["Rand Error %"] = 100 * (comp_df["Random"] - comp_df["Overall"]) / comp_df["Overall"]
comp_df["Strat Error %"] = 100 * (comp_df["Stratified"] - comp_df["Overall"]) / comp_df["Overall"]
display(comp_df.round(3))

# Remove income_cat attribute
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.copy()

### Task 5.1 — Compare Sampling Strategies

Stratified sampling ensures that subgroups (strata) of a critical feature (median income) are proportionally represented in both training and test sets. Random sampling introduces sample bias for key predictors, which skews performance evaluation.

## Part VI — Geographical Visualization

In [ ]:
housing.plot(
    kind="scatter", x="longitude", y="latitude", alpha=0.4,
    s=housing["population"]/100, label="population",
    c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
    figsize=(10, 7)
)
plt.title("California Housing Prices and Population Density")
plt.show()

### Task 6.1 — Interpret the Visualization

1. **Geographical patterns:** Prices relate directly to location and high population density.
2. **High-density areas:** Clusters appear along the coast (Los Angeles, San Francisco Bay Area).
3. **Price variation:** Coastal districts exhibit much higher prices than inland areas.
4. **Utility:** Ocean proximity and urban density serve as strong spatial predictors for house values.

## Part VII — Correlation Analysis

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)
plt.title("Median Income vs Median House Value")
plt.show()

### Task 7.1 — Interpret Correlations

1. **Strongest correlation:** `median_income` (~0.687).
2. **Causation:** No. Correlation indicates linear association, not direct causality.
3. **Weak correlations:** Yes. Features with low linear correlation may still contain important non-linear relationships or work well in combination with other features.
4. **Model selection:** No. Correlation measures simple linear pairs, whereas complex algorithms handle interactions and non-linear patterns.

## Part VIII — Feature Engineering

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

corr_matrix = housing.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

### Task 8.1 — Feature Engineering Discussion

1. **Why combine attributes?** Ratios (e.g., bedrooms per room) provide cleaner, more actionable indicators than raw district totals.
2. **Most promising feature:** `bedrooms_per_room` (~ -0.26 correlation); houses with a lower bedroom-to-total-room ratio are typically more expensive.
3. **Too many features:** Creates high dimensionality, increases computational cost, and risks overfitting.

## Part IX — Handling Missing Values

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

housing_num = housing.drop("ocean_proximity", axis=1)
imputer = SimpleImputer(strategy="median")
imputer.fit(housing_num)

X = imputer.transform(housing_num)
print("Imputer learned medians:\n", pd.Series(imputer.statistics_, index=housing_num.columns))

### Task 9.1 — Compare Missing-Value Strategies

- **Delete rows:** Drops observations; risks throwing away valid data if missingness is rare.
- **Delete column:** Removes the entire feature; loses useful predictive information.
- **Median imputation:** Preserves dataset size and retains feature information without being distorted by outliers.
- **Training set calculation:** Prevents data leakage by ensuring information from the test set does not influence model fitting.

## Part X — Handling Categorical Attributes

In [ ]:
housing_cat = housing[["ocean_proximity"]]
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

print("Encoded categories:", cat_encoder.categories_)
print("One-hot matrix shape:", housing_cat_1hot.shape)

### Task 10.1 — Understand One-Hot Encoding

1. **Why not 1, 2, 3...?** Integer encoding implies a numerical order ($1 < 2 < 3$) where none exists, confusing distance-based algorithms.
2. **Goal of One-Hot Encoding:** Creates binary columns (0 or 1) for each category to treat them independently.
3. **Number of created features:** 5 binary columns.

## Part XI — Feature Scaling

In [ ]:
scaler = StandardScaler()
housing_num_scaled = scaler.fit_transform(housing_num)
print("Scaled Mean:", housing_num_scaled.mean(axis=0).round(2))
print("Scaled Std Dev:", housing_num_scaled.std(axis=0).round(2))

### Task 11.1 — Understand Feature Scaling

1. **Scale issues:** Large numerical ranges dominate gradient updates and distance metrics.
2. **Standardization:** Shifts data to mean 0 and scales to standard deviation 1.
3. **Information loss:** No; it changes scale unit representation without altering underlying data geometry.
4. **Requirement:** Required for linear models, SVMs, and neural networks; optional for decision trees and random forests.

## Part XII — Building a Preprocessing Pipeline

In [ ]:
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

housing_prepared = full_pipeline.fit_transform(housing)
print("Final prepared data shape:", housing_prepared.shape)

### Task 12.1 — Understand the Pipeline

1. **Missing values:** Handled by `SimpleImputer` inside `num_pipeline`.
2. **Scaling:** Handled by `StandardScaler` inside `num_pipeline`.
3. **Categorical encoding:** Handled by `OneHotEncoder` inside `ColumnTransformer`.
4. **Pipeline benefits:** Automates transformations, guarantees clean execution, prevents data leakage, and integrates preprocessing into production.